In [0]:
# ===================================================
# BLOCK 1 — IMPORTS AND CONFIGURATION (PYTHON)
# ===================================================

from pyspark.sql import functions as F

CATALOG = "semiconplus_portfolio"
SILVER_OEE = f"{CATALOG}.silver.simulated_hourly_equipment_operations"
SILVER_EVENTS = f"{CATALOG}.silver.equipment_events"
DIM_DATE = f"{CATALOG}.gold.dim_date"
DIM_SITE = f"{CATALOG}.gold.dim_site"
DIM_EQUIPMENT = f"{CATALOG}.gold.dim_equipment"
FACT_OEE = f"{CATALOG}.gold.fact_oee_hourly"
FACT_EVENTS = f"{CATALOG}.gold.fact_equipment_events_observed"
MART_OEE_DAILY = f"{CATALOG}.gold.mart_equipment_oee_daily"
MART_LOSSES = f"{CATALOG}.gold.mart_equipment_losses_daily"
LOCAL_TIMEZONE = "Asia/Taipei"

spark.conf.set("spark.sql.session.timeZone", "UTC")
print("Day 4 operational fact configuration loaded.")

In [0]:
# ===================================================
# BLOCK 2 — BUILD THE SYNTHETIC HOURLY OEE FACT (PYTHON)
# ===================================================

oee_fact_df = (
    spark.table(SILVER_OEE).alias("source")
    .join(spark.table(DIM_DATE).alias("date"), F.col("source.production_date") == F.col("date.full_date"), "left")
    .join(spark.table(DIM_SITE).alias("site"), F.col("source.site_id") == F.col("site.site_id"), "left")
    .join(spark.table(DIM_EQUIPMENT).alias("equipment"), F.col("source.equipment_id") == F.col("equipment.equipment_id"), "left")
    .select(
        F.col("source.operation_record_id"), F.col("source.operation_hour_utc"),
        F.col("source.operation_hour_local"), F.col("date.date_key"),
        F.col("source.production_date"), F.col("site.site_key"),
        F.col("equipment.equipment_key"), F.col("source.site_id"),
        F.col("source.equipment_id"), F.col("source.scheduled_time_seconds"),
        F.col("source.approved_planned_downtime_seconds"),
        F.col("source.planned_production_time_seconds"),
        F.col("source.unplanned_downtime_seconds"), F.col("source.operating_time_seconds"),
        F.col("source.short_stop_seconds"), F.col("source.setup_seconds"),
        F.col("source.run_seconds"), F.col("source.total_units"), F.col("source.good_units"),
        F.col("source.rated_units_per_hour"), F.col("source.theoretical_output_units"),
        F.when(F.col("source.planned_production_time_seconds") > 0,
               F.col("source.operating_time_seconds") / F.col("source.planned_production_time_seconds")).alias("availability"),
        F.when(F.col("source.theoretical_output_units") > 0,
               F.col("source.total_units") / F.col("source.theoretical_output_units")).alias("utilization"),
        F.when(F.col("source.total_units") > 0,
               F.col("source.good_units") / F.col("source.total_units")).alias("quality"),
        F.col("source.simulation_seed"), F.col("source.simulation_version"),
        F.col("source.simulated_record_flag"), F.col("source.record_origin"),
    )
    .withColumn("oee", F.col("availability") * F.col("utilization") * F.col("quality"))
    .withColumn("oee_target", F.lit(0.80))
    .withColumn("oee_target_met_flag", F.col("oee") >= F.col("oee_target"))
    .withColumn("_gold_processed_at_utc", F.current_timestamp())
)

unresolved_oee_keys = oee_fact_df.filter(F.col("date_key").isNull() | F.col("site_key").isNull() | F.col("equipment_key").isNull()).count()
assert unresolved_oee_keys == 0
(oee_fact_df.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(FACT_OEE))
print(f"Synthetic hourly OEE fact rows: {spark.table(FACT_OEE).count():,}")

In [0]:
# ===================================================
# BLOCK 3 — BUILD THE OBSERVED EQUIPMENT-EVENT FACT (PYTHON)
# ===================================================

observed_events_df = (
    spark.table(SILVER_EVENTS).alias("source")
    .withColumn("event_timestamp_local", F.from_utc_timestamp("event_timestamp_utc", LOCAL_TIMEZONE))
    .withColumn("production_date", F.to_date(F.col("event_timestamp_local") - F.expr("INTERVAL 8 HOURS")))
    .withColumn(
        "event_end_timestamp_utc",
        F.expr(
            "event_timestamp_utc + "
            "make_interval(0, 0, 0, 0, 0, 0, duration_seconds)"
        ),
    )
    .withColumn(
        "event_classification",
        F.when((F.col("event_type") == "UNPLANNED_DOWNTIME") & (F.col("duration_seconds") > 300), "DOWNTIME")
         .when((F.col("event_type") == "UNPLANNED_DOWNTIME") & (F.col("duration_seconds") <= 300), "SHORT_STOP")
         .when((F.col("event_type") == "PLANNED_DOWNTIME") & (F.col("duration_seconds") > 3600), "PLANNED_OVERRUN")
         .when(F.col("event_type") == "PLANNED_DOWNTIME", "PLANNED_DOWNTIME")
         .when(F.col("event_type") == "ALARM", "ALARM")
         .otherwise(F.col("event_type")),
    )
    .join(spark.table(DIM_DATE).alias("date"), F.col("production_date") == F.col("date.full_date"), "left")
    .join(spark.table(DIM_SITE).alias("site"), F.col("source.site_id") == F.col("site.site_id"), "left")
    .join(spark.table(DIM_EQUIPMENT).alias("equipment"), F.col("source.equipment_id") == F.col("equipment.equipment_id"), "left")
    .select(
        F.sha2(F.concat_ws("|", F.lit("OBSERVED_EVENT_V1"), F.col("source.event_id")), 256).alias("event_key"),
        F.col("source.event_id"), F.col("date.date_key"), F.col("production_date"),
        F.col("site.site_key"), F.col("equipment.equipment_key"),
        F.col("source.site_id"), F.col("source.equipment_id"),
        F.col("source.event_timestamp_utc"), F.col("event_timestamp_local"),
        F.col("event_end_timestamp_utc"), F.col("source.event_type"),
        F.col("event_classification"), F.col("source.duration_seconds"),
        F.col("source.alarm_code"), F.col("source.source_system"),
        F.lit(False).alias("simulated_record_flag"),
        F.lit("OBSERVED_SILVER_EQUIPMENT_EVENTS").alias("record_origin"),
        F.current_timestamp().alias("_gold_processed_at_utc"),
    )
)

unresolved_event_keys = observed_events_df.filter(F.col("date_key").isNull() | F.col("site_key").isNull() | F.col("equipment_key").isNull()).count()
assert unresolved_event_keys == 0
(observed_events_df.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(FACT_EVENTS))
print(f"Observed equipment-event fact rows: {spark.table(FACT_EVENTS).count():,}")

In [0]:
# ===================================================
# BLOCK 4 — BUILD DAILY WEIGHTED OEE MART (PYTHON)
# ===================================================

daily_df = (
    spark.table(FACT_OEE)
    .groupBy("date_key", "production_date", "site_key", "equipment_key", "site_id", "equipment_id")
    .agg(
        F.sum("scheduled_time_seconds").alias("scheduled_time_seconds"),
        F.sum("approved_planned_downtime_seconds").alias("approved_planned_downtime_seconds"),
        F.sum("planned_production_time_seconds").alias("planned_production_time_seconds"),
        F.sum("unplanned_downtime_seconds").alias("unplanned_downtime_seconds"),
        F.sum("operating_time_seconds").alias("operating_time_seconds"),
        F.sum("short_stop_seconds").alias("short_stop_seconds"),
        F.sum("setup_seconds").alias("setup_seconds"),
        F.sum("theoretical_output_units").alias("theoretical_output_units"),
        F.sum("total_units").alias("total_units"), F.sum("good_units").alias("good_units"),
        F.first("simulation_seed").alias("simulation_seed"),
        F.first("simulation_version").alias("simulation_version"),
        F.first("simulated_record_flag").alias("simulated_record_flag"),
    )
    .withColumn("availability", F.col("operating_time_seconds") / F.col("planned_production_time_seconds"))
    .withColumn("utilization", F.col("total_units") / F.col("theoretical_output_units"))
    .withColumn("quality", F.col("good_units") / F.col("total_units"))
    .withColumn("oee", F.col("availability") * F.col("utilization") * F.col("quality"))
    .withColumn("oee_target", F.lit(0.80))
    .withColumn("oee_target_met_flag", F.col("oee") >= F.col("oee_target"))
    .withColumn("_gold_processed_at_utc", F.current_timestamp())
)
(daily_df.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(MART_OEE_DAILY))
print(f"Daily OEE mart rows: {spark.table(MART_OEE_DAILY).count():,}")

In [0]:
# ===================================================
# BLOCK 5 — BUILD DAILY EQUIPMENT-LOSS MART (PYTHON)
# ===================================================

loss_df = daily_df.select(
    "date_key", "production_date", "site_key", "equipment_key", "site_id", "equipment_id",
    "approved_planned_downtime_seconds", "unplanned_downtime_seconds",
    "short_stop_seconds", "setup_seconds",
    (F.col("total_units") - F.col("good_units")).alias("reject_units"),
    "simulation_seed", "simulation_version", "simulated_record_flag",
    F.lit("DETERMINISTIC_PORTFOLIO_SIMULATION").alias("record_origin"),
    F.current_timestamp().alias("_gold_processed_at_utc"),
)
(loss_df.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(MART_LOSSES))

summary = [(FACT_OEE, spark.table(FACT_OEE).count()),
           (FACT_EVENTS, spark.table(FACT_EVENTS).count()),
           (MART_OEE_DAILY, spark.table(MART_OEE_DAILY).count()),
           (MART_LOSSES, spark.table(MART_LOSSES).count())]
display(spark.createDataFrame(summary, ["table_name", "row_count"]))